# 00 — Pull Settled Market Metadata

Pulls lightweight metadata for **every settled Kalshi market** and saves a joined,
flat CSV to `../data/markets_metadata.csv`. This file feeds the next notebook,
which algorithmically selects matched sports / political market pairs.

## Key API notes

| Query param | Internal `status` returned | Has result? | Meaning |
|---|---|---|---|
| `status=settled` | `finalized` | yes | **Fully resolved & paid out — what we want** |
| `status=closed` | `closed` | no | Past close_time, outcome not yet known |
| `status=closed` | `determined` | yes | Outcome known, payout not processed yet |
| `status=finalized` | *(returns 0 results)* | — | Not a valid query param |

`status=settled` is the correct query param. It maps exactly to `finalized` records
internally — the same status seen on the Warsh Fed Chair market we already pulled.

## Schema split

The Kalshi API splits metadata across two endpoints:
- **`/markets`** — volume, dates, result, ticker, title — **no `category`**
- **`/events`** — category, series_ticker — **no volume**

Both are paginated with a cursor. We first pull all events, as they are fewer than markets and have the Category tag. We then filter down to the correct categories, before using that finalized list of qualifying events to pull only the necessary markets.


## Cache pattern

If `../data/markets_metadata.csv` already exists the notebook loads it and skips
both API pulls. Delete that file (or rename it) to force a fresh pull.

## 1. Setup

In [1]:
import requests
import pandas as pd
import time
import os

BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"
DATA_DIR = "../data/"
METADATA_PATH = os.path.join(DATA_DIR, "markets_metadata.csv")




print(f"Data directory : {os.path.abspath(DATA_DIR)}")
print(f"Output file    : {os.path.abspath(METADATA_PATH)}")
print(f"Cache exists   : {os.path.exists(METADATA_PATH)}")

Data directory : /Users/josephsemelroth/Desktop/QSS 20 Final Project/data
Output file    : /Users/josephsemelroth/Desktop/QSS 20 Final Project/data/markets_metadata.csv
Cache exists   : True


## 2. Helper functions

In [2]:
   """
    Paginate GET /markets?status=settled and return all records as a DataFrame.

    Uses the same cursor pattern as get_historical_trades() in pull_data.ipynb.
    Each page returns up to 1,000 market records. Fields include:
        ticker, event_ticker, title, open_time, close_time, settlement_ts,
        volume_fp, result, status, market_type, and ~30 others.
    Note: 'category' is NOT present here — it lives on the /events endpoint.

    Parameters
    ----------
    max_pages : int or None
        Cap the number of pages fetched (useful for testing). None = no cap.
    verbose : bool
        Print progress after each page.

    Returns
    -------
    pd.DataFrame
    """

def get_all_settled_markets(max_pages=None, verbose=True):
 
    all_markets = []
    cursor = None
    page = 0

    while True:
        params = {"status": "settled", "limit": 1000}
        if cursor:
            params["cursor"] = cursor

        r = requests.get(f"{BASE_URL}/markets", params=params)
        if r.status_code != 200:
            print(f"Error {r.status_code}: {r.text[:300]}")
            break

        data = r.json()
        markets = data.get("markets", [])
        all_markets.extend(markets)
        page += 1

        if verbose:
            print(f"  page {page:>3}: +{len(markets):,} markets  (running total: {len(all_markets):,})")

        cursor = data.get("cursor")
        if not cursor or len(markets) == 0:
            break
        if max_pages and page >= max_pages:
            break

        time.sleep(REQUEST_DELAY)

    print(f"  done: {len(all_markets):,} total market records")
    return pd.DataFrame(all_markets)


def get_all_settled_events(max_pages=None, verbose=True):
    """
    Paginate GET /events?status=settled and return all records as a DataFrame.
    Retries automatically on 429 rate-limit responses with exponential backoff.
    """
    all_events = []
    cursor = None
    page = 0

    while True:
        params = {"status": "settled", "limit": 200}   # /events max is 200
        if cursor:
            params["cursor"] = cursor

        # Retry loop for 429 rate-limit errors
        backoff = 5
        for attempt in range(5):
            r = requests.get(f"{BASE_URL}/events", params=params)
            if r.status_code == 429:
                print(f"  429 rate limit — waiting {backoff}s before retry {attempt+1}/5 ...")
                time.sleep(backoff)
                backoff *= 2
            else:
                break

        if r.status_code != 200:
            print(f"Error {r.status_code}: {r.text[:300]}")
            break

        data = r.json()
        events = data.get("events", [])
        all_events.extend(events)
        page += 1

        if verbose:
            print(f"  page {page:>3}: +{len(events):,} events  (running total: {len(all_events):,})")

        cursor = data.get("cursor")
        if not cursor or len(events) == 0:
            break
        if max_pages and page >= max_pages:
            break

        time.sleep(REQUEST_DELAY)

    print(f"  done: {len(all_events):,} total event records")
    return pd.DataFrame(all_events)



def get_markets_for_event(event_ticker):
    """
    Fetch all settled markets for a single event_ticker via /historical/markets.
    This endpoint covers older settled markets not accessible via /markets.
    Returns a list of market dicts (caller aggregates into a DataFrame).
    """
    all_markets = []
    cursor = None

    while True:
        params = {"event_ticker": event_ticker, "limit": 1000}
        if cursor:
            params["cursor"] = cursor

        r = requests.get(f"{BASE_URL}/historical/markets", params=params)
        if r.status_code != 200:
            break

        data = r.json()
        markets = data.get("markets", [])
        all_markets.extend(markets)

        cursor = data.get("cursor")
        if not cursor or len(markets) == 0:
            break

        time.sleep(REQUEST_DELAY)

    return all_markets

print("Helper functions defined.")

Helper functions defined.


## 4. Full paginated pull (cache-aware)

**Cache logic:**
- If `../data/markets_metadata.csv` exists → load it, skip all API calls
- If not → pull all pages from both endpoints, join on `event_ticker`, save

Delete `markets_metadata.csv` to force a fresh pull.

In [3]:
if os.path.exists(METADATA_PATH):
    # ── Cache hit ──────────────────────────────────────────────────────────────
    print(f"Cache hit — loading from {METADATA_PATH}")
    df = pd.read_csv(METADATA_PATH, low_memory=False)
    print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")

else:
    # ── Cache miss: events-first targeted pull ─────────────────────────────────
    print("Cache miss — pulling from Kalshi API (events-first approach) ...")
    print()

    # Stage 1: pull all events to get categories
    print("Stage 1 of 3: /events?status=settled")
    events_raw = get_all_settled_events(max_pages=17, verbose=True)
    events_slim = (
        events_raw[["event_ticker", "series_ticker", "category"]]
        .drop_duplicates(subset="event_ticker")
    )
    print(f"  unique event_tickers: {len(events_slim):,}")
    print()

    # Stage 2: filter to relevant categories only
    RELEVANT_CATEGORIES = {"Sports", "Politics", "Elections"}
    qualifying = (
        events_slim[events_slim["category"].isin(RELEVANT_CATEGORIES)]
        .reset_index(drop=True)
    )
    print(f"Stage 2 of 3: keeping {RELEVANT_CATEGORIES}")
    for cat, grp in qualifying.groupby("category"):
        print(f"  {cat}: {len(grp):,} events")
    print(f"  total qualifying events: {len(qualifying):,}")
    print()

    # Stage 3: pull markets for each qualifying event
    print(f"Stage 3 of 3: pulling markets for {len(qualifying):,} events ...")
    all_markets = []
    for i, row in qualifying.iterrows():
        if i % 50 == 0:
            print(f"  {i:>4}/{len(qualifying)}  events processed  ({len(all_markets):,} markets so far)")
        markets = get_markets_for_event(row["event_ticker"])
        for m in markets:
            m["_category"] = row["category"]
        all_markets.extend(markets)
        time.sleep(REQUEST_DELAY)

    print(f"  done: {len(all_markets):,} total market records")
    print()

    if not all_markets:
        raise RuntimeError("No markets fetched — check API connectivity.")

    markets_df = pd.DataFrame(all_markets)

    # Stage 4: derive analysis_category from the category carried through
    def _map_category(c):
        if c == "Sports":
            return "Sports"
        if c in ("Politics", "Elections"):
            return "Political"
        return "Other"
    markets_df["analysis_category"] = markets_df["_category"].map(_map_category)

    # Stage 5: KXMVE safety filter (should be zero rows at this point)
    n_before = len(markets_df)
    markets_df = markets_df[~markets_df["event_ticker"].str.startswith("KXMVE", na=False)].copy()
    removed = n_before - len(markets_df)
    if removed:
        print(f"  WARNING: KXMVE filter removed {removed:,} unexpected rows")

    # Stage 6: trim to final column set
    KEEP_COLS = ["ticker", "event_ticker", "title", "open_time", "close_time",
                 "settlement_ts", "volume_fp", "result", "status", "analysis_category"]
    df = markets_df[[c for c in KEEP_COLS if c in markets_df.columns]]
    print(f"  trimmed to {len(df.columns)} columns: {list(df.columns)}")
    print()

    # Save
    os.makedirs(DATA_DIR, exist_ok=True)
    df.to_csv(METADATA_PATH, index=False)
    print(f"Saved {len(df):,} rows -> {os.path.abspath(METADATA_PATH)}")

print("\nDone.")


Cache hit — loading from ../data/markets_metadata.csv
Loaded 3,393 rows, 10 columns

Done.


## 5. Verification

Confirm that our two anchor markets — Indiana CFP championship (`KXNCAAF-26`) and
Trump's Fed Chair nomination of Kevin Warsh (`KXFEDCHAIRNOM-29`) — appear in the
dataset with the expected `analysis_category` values.

In [4]:
# ── Post-pull verification ────────────────────────────────────────────────────
ANCHOR_EVENTS = {
    "KXNCAAF-26":      "Sports",
    "KXFEDCHAIRNOM-29": "Political",
}

print("=" * 60)
print("Verification: anchor markets present with correct categories")
print("=" * 60)

all_passed = True
for event_ticker, expected_cat in ANCHOR_EVENTS.items():
    matches = df[df["event_ticker"] == event_ticker]
    if matches.empty:
        print(f"  FAIL  {event_ticker!r:30} — NOT FOUND in dataset")
        all_passed = False
    else:
        actual_cats = matches["analysis_category"].unique().tolist()
        ok = all(c == expected_cat for c in actual_cats)
        status = "PASS" if ok else "FAIL"
        if not ok:
            all_passed = False
        print(f"  {status}  {event_ticker!r:30} — expected={expected_cat!r:12}  got={actual_cats}")

print()
if all_passed:
    print("All checks passed.")
else:
    print("One or more checks FAILED — inspect above.")


Verification: anchor markets present with correct categories
  PASS  'KXNCAAF-26'                   — expected='Sports'      got=['Sports']
  PASS  'KXFEDCHAIRNOM-29'             — expected='Political'   got=['Political']

All checks passed.
